In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error,  mean_squared_error
from sklearn.neighbors import NearestNeighbors

In [20]:
df_menu = pd.read_csv('kombinasi_menu.csv')
df_menu.head()

,Id_kombinasi,Makan_pagi,Makan_siang,Makan_malam,Snack_1,Snack_2,Total_natrium
0,1,Kering tempe,Tumis buncis wortel bintik,Tuna sambal woku,Susu,Buah delima,681.0
1,2,Rawon,Mie ayam,Bubur manado,Sari buah tropika,Buah delima,701.0
2,3,Coto makasar,Tumis buncis wortel bintik,Makaroni bumbu merah,Sup buah serut,Kue sus,687.3
3,4,Nasi pecel,Sate kambing,Sambal udang,Jus wortel tomat,Semangka,460.0
4,5,Teri basah goreng renyah,Bakmi,Urab,Apel,Kue nagasari,512.0


In [ ]:
def get_kategori_natrium(tekanan_darah):
    if isinstance(tekanan_darah, str):
        try:
            sistolik, diastolik = map(int, tekanan_darah.split('/'))
        except:
            raise ValueError("Format tekanan darah harus seperti '180/110'")
    elif isinstance(tekanan_darah, (tuple, list)) and len(tekanan_darah) == 2:
        sistolik, diastolik = tekanan_darah
    else:
        raise ValueError("Tekanan darah harus dalam format '180/110' atau tuple (sistolik, diastolik)")
    
    def konversi_ke_sendok_teh(rentang):
        return round(rentang[0]/ 2000, 2), round(rentang[1] / 2000, 2)
    
    if 140 <= sistolik <= 159 or 90 <= diastolik <= 99:
        batas_natrium = (1000, 1200)
        batas_sdt = konversi_ke_sendok_teh(batas_natrium)
        return "Hipertensi Derajat 1", batas_natrium, batas_sdt
    elif 160 <= sistolik <= 179 or 100 <= diastolik <= 109:
        batas_natrium = (600, 800)
        batas_sdt = konversi_ke_sendok_teh(batas_natrium)
        return "Hipertensi Derajat 2", batas_natrium, batas_sdt
    elif sistolik >= 180 or diastolik >= 110:
        batas_natrium = (200, 400)
        batas_sdt = konversi_ke_sendok_teh(batas_natrium)
        return "Hipertensi Derajat 3", batas_natrium, batas_sdt
    else:
        raise ValueError("Tekanan darah tidak sesuai dengan kategori hipertensi.")
    
def penjelasan_sdt_praktis(min_sdt, max_sdt):
    rata_rata = (min_sdt + max_sdt) / 2
    if rata_rata <= 0.2:
        return "setara 1/10 sendok teh garam"
    elif rata_rata <= 0.45:
        return "setara 1/3 sendok teh garam"
    elif rata_rata <= 0.6:
        return "setara 1/2 sendok teh garam"
    else:
        return "perkiraan sesuai kebutuhan harian"

SyntaxError: invalid syntax (1197853547.py, line 18)

In [ ]:
def rekomendasi_menu(tekanan_darah, preferensi= None, max_k=50, min_hasil=1):

    #ambil rentang natrium berdasarkan tekanan darah
    kategori, (min_natrium, max_natrium) , (min_sdt, max_sdt) = get_kategori_natrium(tekanan_darah)
    takaran_praktis = penjelasan_sdt_praktis(min_sdt, max_sdt)
    target_natrium = (min_natrium + max_natrium) / 2

    #baca data kombinasi menu
    df_menu = pd.read_csv('kombinasi_menu.csv')
    X = df_menu[['Total_natrium']].values.reshape(-1, 1)


    for k in range(1, max_k + 1):
        model = NearestNeighbors(n_neighbors = k, metric='euclidean').fit(X)
        _, indices = model.kneighbors([[target_natrium]])
        hasil = df_menu.iloc[indices[0]]

    #filter jika ada preferensi makan
    if preferensi:
        hasil = hasil[
            hasil.apply(
                lambda row: any(pref.lower() in row.to_string().lower() for pref in preferensi), axis=1
            )
        ]
        
        if len(hasil) >= min_hasil:
            hasil['Selisih'] = abs(hasil['Total_natrium'] - target_natrium)
            hasil = (
                hasil
                .sort_values('Selisih')
                .drop(columns='Selisih')
                .assign(Total_natrium=lambda df: df['Total_natrium'].round(2))
                .reset_index(drop=True)
            )
            return hasil

    # 6. Jika tidak ada k yang memenuhi, return DataFrame kosong
    return pd.DataFrame(columns=df_menu.columns)


In [ ]:
#input pasien
tekanan_darah = ("180/110")
preferensi_pasien = ['ikan']

#panggil fungsi rekomendasi
hasil = rekomendasi_menu(tekanan_darah, preferensi=preferensi_pasien )
print(hasil)

   Id_kombinasi           Makan_pagi               Makan_siang  \
0           698  Ikan tongkol goreng        Bening bayam tauge   
1           396    Tempe masak wijen          Ikan kuah kuning   
2            22          Sayur pakis        Sayur bunga pepaya   
3           107       Ayam rica-rica  Ikan panggang siram acar   
4           785          Nasi kebuli         Ikan patin goreng   
5           559   Bubur kacang hijau         Ikan patin goreng   
6           892    Ikan bawal goreng                   Rendang   
7           137      Ikan mas goreng                 Ayam woku   

           Makan_malam            Snack_1       Snack_2  Total_natrium  
0     Sayur rica rodoh               Apel       Es krim          280.1  
1           Tahu siksa         Susu murni  Susu kedelai          272.0  
2          Gulai ikan           Talam ubi     Jus jeruk          270.3  
3    Ikan mujair pepes       Jus stroberi         Ronde          331.7  
4  Kangkung bumbu kare             Klepo

In [ ]:
cases = [
    ("180/110", ["ikan"]),
    ("150/95", ["sayur"]),
    ("165/106", ["ayam"]),
    ("145/92", ["daging"]),
    ("170/101", ["ikan panggang"]),
    ("160/103", ["nasi goreng"]),
    ("155/98", ["ikan", "sayur"]),
    ("140/96", ["ayam", "daging"]),
    ("177/106", ["tekwan"]),
    ("156/95", ["dendeng"]),
    ("165/107", ["ikan", "durian"]),
    ("149/92", ["sup"]),
    ("178/105", ["telur"]),
    ("167/108", ["tahu bacem"]),
    ("155/99", ["ayam goreng"]),
    ("140/90", ["ikan kembung"]),
    ("175/102", ["nasi goreng"]),
    ("189/110", ["udang"]),
    ("150/93", ["ikan mas"]),
    ("165/100", ["pepes"]),
    ("146/99", ["semur jengkol"]),
    ("170/101", ["batagor"]),
    ("179/100", ["balado"]),
    ("159/98", ["daging sapi"]),
    ("144/90", ["Gulai ikan"]),
    ("175/103", ["nasi goreng"]),
    ("180/120", ["kacang panjang"]),
    ("165/97", ["ketoprak"]),
    ("150/97", ["sate"]),
    ("170/108", ["teri"]),
]

y_true = []
y_pred = []

for tekanan_darah, preferensi in cases:
    hasil = rekomendasi_menu(tekanan_darah, preferensi)
    
    if hasil.empty:
        print(f"{tekanan_darah}, {preferensi} → ❌ Tidak ada hasil")
    else:
        kategori, (min_natrium, max_natrium) , (min_sdt, max_sdt) = get_kategori_natrium(tekanan_darah)
        target_natrium = (min_natrium + max_natrium) / 2
        rata2 = hasil['Total_natrium'].mean()
        selisih = abs(rata2 - target_natrium)
        print(f"{tekanan_darah}, {preferensi} → ✅ {len(hasil)} hasil, rata-rata natrium: {rata2:.0f} mg, selisih: {selisih:.0f} mg")

        y_true.append(target_natrium)
        y_pred.append(rata2)


180/110, ['ikan'] → ✅ 8 hasil, rata-rata natrium: 319 mg, selisih: 19 mg
150/95, ['sayur'] → ✅ 6 hasil, rata-rata natrium: 1109 mg, selisih: 9 mg
165/106, ['ayam'] → ✅ 18 hasil, rata-rata natrium: 697 mg, selisih: 3 mg
145/92, ['daging'] → ✅ 3 hasil, rata-rata natrium: 1098 mg, selisih: 2 mg
170/101, ['ikan panggang'] → ✅ 2 hasil, rata-rata natrium: 711 mg, selisih: 11 mg
160/103, ['nasi goreng'] → ✅ 1 hasil, rata-rata natrium: 673 mg, selisih: 27 mg
155/98, ['ikan', 'sayur'] → ✅ 16 hasil, rata-rata natrium: 1103 mg, selisih: 3 mg
140/96, ['ayam', 'daging'] → ✅ 25 hasil, rata-rata natrium: 1101 mg, selisih: 1 mg
177/106, ['tekwan'] → ❌ Tidak ada hasil
156/95, ['dendeng'] → ✅ 2 hasil, rata-rata natrium: 1083 mg, selisih: 17 mg
165/107, ['ikan', 'durian'] → ✅ 18 hasil, rata-rata natrium: 692 mg, selisih: 8 mg
149/92, ['sup'] → ✅ 3 hasil, rata-rata natrium: 1106 mg, selisih: 6 mg
178/105, ['telur'] → ✅ 2 hasil, rata-rata natrium: 685 mg, selisih: 15 mg
167/108, ['tahu bacem'] → ✅ 1 hasil,

In [ ]:
mae = mean_absolute_error(y_true, y_pred)
rmse= np.sqrt (mean_squared_error (y_true, y_pred))

print("\n📈 Evaluasi Model:")
print("MAE:", round(mae, 2), "mg")
print("RMSE:", round(rmse, 2), "mg")



📈 Evaluasi Model:
MAE: 10.15 mg
RMSE: 13.28 mg


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def uji_k_berbagai_jarak(df_menu, target_natrium, preferensi=None, max_k=50, metrics=['euclidean', 'manhattan', 'cosine']):
    hasil_semua = []

    for metric in metrics:
        for k in range(1, max_k+1):
            # Data input hanya kolom Total_natrium
            X = df_menu[['Total_natrium']].values.reshape(-1, 1)
            model = NearestNeighbors(n_neighbors=k, metric=metric)
            model.fit(X)
            _, indices = model.kneighbors([[target_natrium]])

            rekom = df_menu.iloc[indices[0]]

            # Filter preferensi (jika ada)
            if preferensi:
                rekom = rekom[
                    rekom.apply(lambda row: any(pref.lower() in row.to_string().lower() for pref in preferensi), axis=1)
                ]
                if rekom.empty:
                    continue  # Skip kalau tidak ada hasil

            # Hitung rata-rata natrium hasil rekomendasi
            rata_natrium = rekom['Total_natrium'].mean()
            rmse = root_mean_squared_error([target_natrium]*len(rekom), rekom['Total_natrium'])

            mae = mean_absolute_error([target_natrium]*len(rekom), rekom['Total_natrium'])

            hasil_semua.append({
                'k': k,
                'metric': metric,
                'jumlah_rekomendasi': len(rekom),
                'rata_natrium': round(rata_natrium, 2),
                'RMSE': round(rmse, 2),
                'MAE': round(mae, 2)
            })

    return pd.DataFrame(hasil_semua)


In [ ]:
# Contoh
tekanan_darah = '170/110'
preferensi = ['ikan']

# Ambil natrium target dari tekanan darah
_, (min_n, max_n), (min_sdt, max_sdt) = get_kategori_natrium(tekanan_darah)
target_n = (min_n + max_n) / 2

# Baca data menu
df = pd.read_csv('kombinasi_menu.csv')

# Jalankan fungsi
hasil = uji_k_berbagai_jarak(df, target_n, preferensi=preferensi, max_k=50)

# Tampilkan k terbaik per metric
print(hasil.loc[hasil.groupby('metric')['RMSE'].idxmin()])


     k     metric  jumlah_rekomendasi  rata_natrium  RMSE   MAE
100  2     cosine                   1         759.0  59.0  59.0
0    1  euclidean                   1         699.7   0.3   0.3
50   1  manhattan                   1         699.7   0.3   0.3
